# Feature Engineering de `merged_datasets`
En esta notebook realizamos la limpieza y la unificación inicial de etiquetas para crear el dataset `data_model`.

- Eliminamos eventos de tipo `TRBA`, `ICEQUAKE` y `REGIONAL`.
- Fusionamos todas las variantes de `VLP` bajo la etiqueta única `VLP`.
- Guardamos el dataset procesado para el modelado posterior.

In [1]:
# Importar librerías y cargar el dataset
import json
import os
import pandas as pd
import numpy as np

path = os.path.join(r'c:\Users\ricar\OneDrive\Desktop\TESIS', 'merged_datasets.json')
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print('shape:', df.shape)
print('\nDistribución original de Type:')
print(df['Type'].value_counts())

shape: (23827, 85)

Distribución original de Type:
Type
LP           12597
VT            8857
TREMI          712
HB             541
TRE            252
VLP            211
VLP_TYPE1      205
TRESP          185
VLP_TYPE2      132
EXPL            87
REGIONAL        27
TRBA            14
ICEQUAKE         7
Name: count, dtype: int64


In [2]:
# Eliminar eventos TRBA, ICEQUAKE y REGIONAL
remove_types = ['TRBA', 'ICEQUAKE', 'REGIONAL']
initial_count = len(df)
df = df[~df['Type'].isin(remove_types)].copy()
removed_count = initial_count - len(df)
print(f'Filas eliminadas: {removed_count}')
print('Nueva forma:', df.shape)
print('\nDistribución de Type después de eliminar clases específicas:')
print(df['Type'].value_counts())

Filas eliminadas: 48
Nueva forma: (23779, 85)

Distribución de Type después de eliminar clases específicas:
Type
LP           12597
VT            8857
TREMI          712
HB             541
TRE            252
VLP            211
VLP_TYPE1      205
TRESP          185
VLP_TYPE2      132
EXPL            87
Name: count, dtype: int64


In [3]:
# Fusionar todas las variantes VLP en la etiqueta 'VLP'
vlp_mask = df['Type'].astype(str).str.upper().str.startswith('VLP', na=False)
df.loc[vlp_mask, 'Type'] = 'VLP'

print('\nDistribución de Type tras unificar VLP:')
print(df['Type'].value_counts())


Distribución de Type tras unificar VLP:
Type
LP       12597
VT        8857
TREMI      712
VLP        548
HB         541
TRE        252
TRESP      185
EXPL        87
Name: count, dtype: int64


In [4]:
# Crear data_model y comprobar composición
data_model = df.copy().reset_index(drop=True)

print('data_model shape:', data_model.shape)
print('\nPrimeras filas de data_model:')
print(data_model.head())

print('\nConteo por Type en data_model:')
print(data_model['Type'].value_counts())

print('\nInformación de data_model:')
print(data_model.info())

print('\nNulos por columna:')
print(data_model.isna().sum())

data_model shape: (23779, 85)

Primeras filas de data_model:
   Type  f_90_percent_energy  f_PeaksAboveRMSDensity_fun   f_energy  \
0  EXPL             7.812500                    0.064327  88.667131   
1  EXPL             7.080078                    0.046784  89.716010   
2  EXPL             9.960938                    0.027290  86.073468   
3  EXPL             7.666016                    0.050682  54.215575   
4  EXPL             9.228516                    0.044834  95.018260   

      f_entropy  f_kurtosis     f_mean  f_multiscaleEntropy  f_peak2rms  \
0 -6.668203e+06    2.335528 -38.546551             2.105047    8.962248   
1 -7.224606e+06    1.937270 -40.262434             2.169781   10.486386   
2 -6.918588e+06    3.035101 -38.394380             1.971289   20.620064   
3 -7.645442e+06    2.024171 -41.384352             2.156271    8.903901   
4 -7.519441e+06    2.656138 -39.678304             1.998668   18.769221   

   f_peak_1020_pos  ...  w_t_peak2rms_D4  w_t_peak2rms_D5  w_

In [5]:
# Guardar data_model en disco
output_dir = os.path.join(r'c:\Users\ricar\OneDrive\Desktop\TESIS')

csv_path = os.path.join(output_dir, 'data_model.csv')
json_path = os.path.join(output_dir, 'data_model.json')
pickle_path = os.path.join(output_dir, 'data_model.pkl')

data_model.to_csv(csv_path, index=False)
data_model.to_json(json_path, orient='records', force_ascii=False)
data_model.to_pickle(pickle_path)

print('Guardado data_model en:')
print('-', csv_path)
print('-', json_path)
print('-', pickle_path)

for file_path in [csv_path, json_path, pickle_path]:
    size = os.path.getsize(file_path)
    print(f'  {os.path.basename(file_path)}: {size} bytes')

Guardado data_model en:
- c:\Users\ricar\OneDrive\Desktop\TESIS\data_model.csv
- c:\Users\ricar\OneDrive\Desktop\TESIS\data_model.json
- c:\Users\ricar\OneDrive\Desktop\TESIS\data_model.pkl
  data_model.csv: 34951037 bytes
  data_model.json: 56941124 bytes
  data_model.pkl: 16103602 bytes
